In [1]:
import qr
import importlib
importlib.reload(qr)

<module 'qr' from '/home/egorobi/Courses/VLCS/inbedded/qr.py'>

In [2]:
import math
import numpy as np
import segno
import cv2
import moderngl

In [3]:
def create_qr(content):
    qr = segno.make_qr(content, version=10, mode="byte")
    matrix = qr.matrix
    arr = np.array(matrix, dtype=int)

    qr.save("output/qr.png", scale=10)

In [4]:
qcd = cv2.QRCodeDetector()

import zxingcpp

def read_qr(img):
    results = zxingcpp.read_barcodes(
        img,
        formats=zxingcpp.BarcodeFormat.QRCode
    )
    return [result.bytes for result in results]

In [5]:
from subtitle_manager import *

manager = SubtitleManager()
manager.import_from_srt("potato_chip_subs.srt")
manager.get_ready(1000)

[Subtitle(startTime=960, endTime=2720, sub_string="Judging by the number of cameras he's")]

In [ ]:
%load_ext autoreload
%autoreload 2

import cv2
import segno
import numpy as np
from pathlib import Path
from protocol import *
from subtitle_manager import *
from video_processor_modified import VideoProcessor

SUB_PATH = "potato_chip_subs.srt"
VIDEO_FILE = "potato_chip.mp4"
OUTPUT_PATH = "output/death_note.mp4"

vp = VideoProcessor(VIDEO_FILE, SUB_PATH)
vp.output(OUTPUT_PATH, double_fps=True, use_skip=False)



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Opened: True
FPS: 30.0
Frames: 2707.0
Frame index: 0
Frame index: 1
Frame index: 2
Frame index: 3
Frame index: 4
Frame index: 5
Frame index: 6
Frame index: 7
Frame index: 8
Frame index: 9
Frame index: 10
Frame index: 11
Frame index: 12
Frame index: 13
Frame index: 14
Frame index: 15
Frame index: 16
Frame index: 17
Frame index: 18
Frame index: 19
Frame index: 20
Frame index: 21
Frame index: 22
Frame index: 23
Frame index: 24
Frame index: 25
Frame index: 26
Frame index: 27
Frame index: 28
Frame index: 29
Frame index: 30
Frame index: 31
Frame index: 32
Frame index: 33
Frame index: 34
Frame index: 35
Frame index: 36
Frame index: 37
Frame index: 38
Frame index: 39
Frame index: 40
Frame index: 41
Frame index: 42
Frame index: 43
Frame index: 44
Frame index: 45
Frame index: 46
Frame index: 47
Frame index: 48
Frame index: 49
Frame index: 50
Frame index: 51
Frame index: 52
Frame index: 53
Frame index: 54
Fram

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

Saved 60.00 FPS video to output/death_note.mp4


[out#0/mp4 @ 0x58b68b488680] video:243148kB audio:1409kB subtitle:0kB other streams:0kB global headers:0kB muxing overhead: 0.043166%
size=  244662kB time=00:01:30.19 bitrate=22221.2kbits/s speed= 107x    


In [37]:
import cv2
import numpy as np
import zxingcpp
from protocol import *

# This should be the Irudin camera index
CAMERA_INDEX = 0

cap = cv2.VideoCapture(CAMERA_INDEX)
cap.set(cv2.CAP_PROP_FPS, 60)
# cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*'YUYV'))

if not cap.isOpened():
    raise RuntimeError("Could not open camera")

decoder = MessageDecoder()
previous = None

print("Camera FPS:", cap.get(cv2.CAP_PROP_FPS))

prev_qr = np.zeros((540, 960)).astype(np.uint8)
prev_toggle = False
count = 0

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        hsv = cv2.cvtColor(cv2.resize(frame, (960, 540)), cv2.COLOR_BGR2HSV)
        h, s, v = cv2.split(hsv)
        gray = cv2.bilateralFilter(v, 1, 75, 75)

        if previous is not None:
            # Signed difference preserves bright/dark polarity
            diff = gray.astype(np.float32) - previous.astype(np.float32)

            # Threshold at zero to recover QR modules
            qr = np.where(diff > 0, 255, 0).astype(np.uint8)
            # mean = cv2.addWeighted(qr, 0.5, prev_qr, 0.5, 0.0)
            # if prev_toggle:
            #     prev_toggle = False
            #     prev_qr = qr.copy()
            # else:
            #     prev_qr = cv2.bitwise_not(qr.copy())
            # qr = mean

            # Try both phase orders
            for candidate in [qr, cv2.bitwise_not(qr.copy())]:
                # Using this library because it returns bytes correctly unlike OpenCV
                results = zxingcpp.read_barcodes(
                    candidate,
                    formats=zxingcpp.BarcodeFormat.QRCode
                )

                for result in results:
                    print(result)
                    for message in decoder.push(result.bytes):
                        print("Decoded:", message)

            if count % 30 == 0:
                cv2.imshow("Reconstructed QR", qr)

            # if cv2.waitKey(5000) & 0xFF == ord("q"):
            #     break

        previous = gray.copy()

        # cv2.imshow("Camera", cv2.resize(frame, (960, 540)))

        count += 1

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

finally:
    cap.release()
    cv2.destroyAllWindows()


Camera FPS: 60.0
Decoded: SendSubtitle(currentTime=500, subtitleStartTime=960, subtitleEndTime=2720, sub_string="Judging by the number of cameras he's")
Decoded: SendSubtitle(currentTime=500, subtitleStartTime=960, subtitleEndTime=2720, sub_string="Judging by the number of cameras he's")
Decoded: SendSubtitle(currentTime=2000, subtitleStartTime=2720, subtitleEndTime=4120, sub_string="planted, he's planning to make a")
Decoded: SendSubtitle(currentTime=2000, subtitleStartTime=2720, subtitleEndTime=4120, sub_string="planted, he's planning to make a")
Decoded: SendSubtitle(currentTime=2000, subtitleStartTime=2720, subtitleEndTime=4120, sub_string="planted, he's planning to make a")
Decoded: SendSubtitle(currentTime=2000, subtitleStartTime=2720, subtitleEndTime=4120, sub_string="planted, he's planning to make a")
Decoded: SendSubtitle(currentTime=2000, subtitleStartTime=2720, subtitleEndTime=4120, sub_string="planted, he's planning to make a")
Decoded: SendSubtitle(currentTime=2000, subtit